In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Load CIFAR-10 dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████████████████████████████████████████████████████████████████████████████████| 170M/170M [02:31<00:00, 1.13MB/s]


In [3]:
# Define ANN Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32*32*3, 3000)
        self.fc2 = nn.Linear(3000, 1000)
        self.fc3 = nn.Linear(1000, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.softmax(self.fc3(x))
        return x

In [4]:
# Define CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 64)
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = self.relu(self.fc1(x))
        x = self.softmax(self.fc2(x))
        return x

In [5]:
# Training function
def train_model(model, trainloader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in trainloader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}')


In [6]:
# Instantiate and train ANN
ann_model = ANN()
train_model(ann_model, trainloader)

Epoch 1, Loss: 2.1876023007780696
Epoch 2, Loss: 2.077750020777173
Epoch 3, Loss: 2.034586037516289
Epoch 4, Loss: 1.9999912923864087
Epoch 5, Loss: 1.9748160770482115


In [7]:
# Instantiate and train CNN
cnn_model = CNN()
train_model(cnn_model, trainloader)

Epoch 1, Loss: 2.210334888809477
Epoch 2, Loss: 2.0786284872942873
Epoch 3, Loss: 2.008899210511571
Epoch 4, Loss: 1.9602224281072007
Epoch 5, Loss: 1.9215845496148405


In [8]:
# Evaluate the models
def evaluate_model(model, testloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in testloader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.numpy())
            all_labels.extend(labels.numpy())
    print(classification_report(all_labels, all_preds, target_names=trainset.classes))

In [9]:
print("\nEvaluation for ANN Model")
evaluate_model(ann_model, testloader)


Evaluation for ANN Model
              precision    recall  f1-score   support

    airplane       0.53      0.57      0.55      1000
  automobile       0.54      0.61      0.57      1000
        bird       0.34      0.23      0.27      1000
         cat       0.35      0.28      0.31      1000
        deer       0.48      0.26      0.33      1000
         dog       0.41      0.36      0.39      1000
        frog       0.46      0.64      0.53      1000
       horse       0.43      0.66      0.52      1000
        ship       0.57      0.65      0.60      1000
       truck       0.58      0.48      0.52      1000

    accuracy                           0.47     10000
   macro avg       0.47      0.47      0.46     10000
weighted avg       0.47      0.47      0.46     10000



In [10]:
print("\nEvaluation for CNN Model")
evaluate_model(cnn_model, testloader)


Evaluation for CNN Model
              precision    recall  f1-score   support

    airplane       0.64      0.49      0.56      1000
  automobile       0.69      0.68      0.69      1000
        bird       0.40      0.46      0.43      1000
         cat       0.36      0.46      0.40      1000
        deer       0.54      0.33      0.41      1000
         dog       0.71      0.18      0.28      1000
        frog       0.46      0.81      0.59      1000
       horse       0.66      0.58      0.62      1000
        ship       0.59      0.76      0.66      1000
       truck       0.60      0.64      0.62      1000

    accuracy                           0.54     10000
   macro avg       0.57      0.54      0.53     10000
weighted avg       0.57      0.54      0.53     10000

